In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

import torch
import transformers

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

## Loading the LM

In [ ]:
from src.utils.training_utils import get_device_map

# model_key = "meta-llama/Llama-3.2-3B"
# model_key = "meta-llama/Llama-3.1-8B-Instruct"
# model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "meta-llama/Llama-3.1-405B-Instruct"

# model_key = "google/gemma-2-9b-it"
model_key = "google/gemma-2-27b-it"

# model_key = "openai/gpt-oss-20b"

# model_key = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# model_key = "allenai/OLMo-2-1124-7B-Instruct"
# model_key = "allenai/OLMo-7B-0424-hf"

# model_key = "Qwen/Qwen2-7B"
# model_key = "Qwen/Qwen2.5-14B-Instruct"
# model_key = "Qwen/Qwen2.5-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "Qwen/Qwen3-1.7B"
# model_key = "Qwen/Qwen3-4B"
# model_key = "Qwen/Qwen3-8B"
# model_key = "Qwen/Qwen3-14B"
# model_key = "Qwen/Qwen3-32B"

# device_map = get_device_map(model_key, 30, n_gpus=8)
# device_map

In [ ]:
from src.models import ModelandTokenizer

# from transformers import BitsAndBytesConfig

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    # device_map=device_map,
    device_map="auto",
    # quantization_config = BitsAndBytesConfig(
    #     # load_in_4bit=True
    #     load_in_8bit=True
    # )
    attn_implementation="eager",
)

## Saving the selection data
> For baseline evaluation. So that every LM is evaluated on the same set of data.

In [ ]:
from src.selection.data import (
    SelectOneTask,
    SelectFirstTask,
    SelectLastTask,
    YesNoTask,
    CountingTask,
)
from typing import Literal

##########################################################
prompt_template_idx = 3  # try out different templates
option_style: Literal["single_line", "numbered"] = "single_line"
n_distractors = (
    5  # number of distractors. total options = n_distractors + 1 for SingleOne task
)
##########################################################

# symantic_type = "objects"
# symantic_type = "profession"
# symantic_type = "nationality"
symantic_type = "landmarks"

TASK_CLS = SelectOneTask

select_task = TASK_CLS.load(
    path=os.path.join(env_utils.DEFAULT_DATA_DIR, "selection", f"{symantic_type}.json")
)
select_task.categories

In [ ]:
sample = select_task.get_random_sample(
    mt=mt,
    option_style=option_style,
    prompt_template_idx=prompt_template_idx,
    category="Japan",
    filter_by_lm_prediction=False,
)

print(sample.prompt(), ">>")
print(f'"{mt.tokenizer.decode([sample.ans_token_id])}"')

predict_next_token(
    inputs=sample.prompt(),
    mt=mt,
)

In [ ]:
import random
random.randint(1,3)

In [ ]:
from tqdm.auto import tqdm

################################################################################################
LIMIT = 1024
N_DISTRACTORS = 5
DS_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "selection/baseline", select_task.task_name
)
################################################################################################

os.makedirs(DS_ROOT, exist_ok=True)

evaluation_samples = []
for _ in tqdm(range(LIMIT)):
    sample = select_task.get_random_sample(
        mt=mt,
        option_style=option_style,
        prompt_template_idx=prompt_template_idx,
        n_distractors=N_DISTRACTORS,
        # n_options=random.randint(1, 3),
        filter_by_lm_prediction=False,
    )
    evaluation_samples.append(sample)

with open(os.path.join(DS_ROOT, f"{symantic_type}.json"), "w") as f:
    json.dump(
        [sample.to_dict() for sample in evaluation_samples],
        f,
        indent=4,
    )

## Load Evaluation Samples

In [ ]:
from src.selection.data import (
    SelectOneTask,
    SelectFirstTask,
    SelectLastTask,
    YesNoTask,
    CountingTask,
)
from typing import Literal
from src.selection.data import SelectionSample, YesNoSample, CountingSample
from src.selection.utils import get_first_token_id
from src.selection.data import MCQify_sample, COUNT_STR_MAP

##########################################################
prompt_template_idx = 3  # try out different templates
option_style: Literal["single_line", "numbered"] = "single_line"
# symantic_type = "objects"
# symantic_type = "profession"
# symantic_type = "nationality"
symantic_type = "landmarks"
##########################################################

TASK_CLS = SelectOneTask
select_task = TASK_CLS.load(
    path=os.path.join(env_utils.DEFAULT_DATA_DIR, "selection", f"{symantic_type}.json")
)

DS_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "selection/baseline", select_task.task_name
)

with open(os.path.join(DS_ROOT, f"{symantic_type}.json"), "r") as f:
    raw_samples = json.load(f)

if TASK_CLS == YesNoTask:
    evaluation_samples = [YesNoSample.from_dict(d) for d in raw_samples]
elif TASK_CLS == CountingTask:
    evaluation_samples = [CountingSample.from_dict(d) for d in raw_samples]
else:
    evaluation_samples = [SelectionSample.from_dict(d) for d in raw_samples]

prompt_template = select_task.prompt_templates[prompt_template_idx]

for idx in range(len(evaluation_samples)):
    evaluation_samples[idx].prompt_template = prompt_template
    # evaluation_samples[idx].option_style = option_style
    if isinstance(evaluation_samples[idx], SelectionSample):
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            name=evaluation_samples[idx].answer, tokenizer=mt.tokenizer, prefix=" "
        )
    elif isinstance(evaluation_samples[idx], CountingSample):
        count_str = COUNT_STR_MAP[evaluation_samples[idx].count]
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            name=count_str, tokenizer=mt.tokenizer, prefix=" "
        )
    elif isinstance(evaluation_samples[idx], YesNoSample):
        yes_mode = evaluation_samples[idx].yes
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            "Yes" if yes_mode else "No", tokenizer=mt.tokenizer, prefix=" "
        )
    # evaluation_samples[idx] = MCQify_sample(
    #     sample=evaluation_samples[idx], tokenizer=mt.tokenizer
    # )

sample = evaluation_samples[15]
print(sample.prompt(), ">>", f'"{mt.tokenizer.decode(sample.ans_token_id)}"')

In [ ]:
from src.selection.utils import get_first_token_id, verify_correct_option
from src.selection.data import get_options_for_answer

result = verify_correct_option(
    mt=mt,
    target=sample.ans_token_id,
    options=get_options_for_answer(sample),
    input=sample.prompt(),
    k=10,
)
result

In [ ]:
from tqdm import tqdm

results = []
for sample in tqdm(evaluation_samples):
    is_correct, pred, track = verify_correct_option(
        mt=mt,
        target=sample.ans_token_id,
        options=get_options_for_answer(sample),
        input=sample.prompt(),
    )
    results.append(
        {
            "sample": sample,
            "is_correct": is_correct,
            "predicted_option": pred,
            "track": track,
        }
    )

In [ ]:
import numpy as np

ranks = []
logits = []
for result in results:
    sample = result["sample"]
    cur_rank = result["track"][sample.ans_token_id][0]
    ranks.append(cur_rank)
    logits.append(result["track"][sample.ans_token_id][1].logit)

n_correct = sum([1 for result in results if result["is_correct"]])
accuracy = n_correct / len(results)

ranks = np.array(ranks)
ranks_avg = ranks.mean()
ranks_std = ranks.std()

logits = np.array(logits)
logits_avg = logits.mean()
logits_std = logits.std()

print(
    f"Accuracy: {accuracy*100:.2f}% ({n_correct}/{len(results)}) | Avg. Rank: {ranks_avg:.2f} ± {ranks_std:.2f} | Avg. Logit: {logits_avg:.2f} ± {logits_std:.2f}"
)

In [ ]:
print(sample.prompt())

In [ ]:
failed_cases = [result for result in results if not result["is_correct"]]

In [ ]:
# sample = failed_cases[26]["sample"]
# print(sample.prompt())

In [ ]:
from src.functional import predict_next_token

predict_next_token(
    inputs=sample.prompt(),
    mt=mt,
)

In [ ]:
sample.ans_token_id, mt.tokenizer.decode(sample.ans_token_id)